In [1]:
import requests
from bs4 import BeautifulSoup
import time

# 1. URL de la lista
base_url = "https://letterboxd.com/dave/list/imdb-top-250/"

# 2. Encabezados de la peticion
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
}

# 3. Primera peticion para descubrir cuantas paginas tiene la lista
response = requests.get(base_url, headers=headers, timeout=30)
print("Codigo HTTP pagina 1:", response.status_code)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Descubrir la paginacion desde el propio HTML (no hace nada en disco)
paginas = [base_url]
for a in soup.select("div.paginate-pages ul li a"):
    href = a.get("href")
    if href:
        paginas.append("https://letterboxd.com" + href)

print("Paginas encontradas:", len(paginas))
for p in paginas:
    print(" -", p)

# 4. Recorrer todas las paginas y acumular las peliculas en memoria
peliculas = []

for numero_pagina, url_pagina in enumerate(paginas, start=1):
    if numero_pagina > 1:
        response = requests.get(url_pagina, headers=headers, timeout=30)
        print("Codigo HTTP pagina", numero_pagina, ":", response.status_code)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

    elementos = soup.select(
        'div.react-component[data-component-class="LazyPoster"]'
    )
    print("Pagina", numero_pagina, ":", len(elementos), "peliculas")

    for indice_local, elemento in enumerate(elementos):
        titulo = elemento.get("data-item-name")
        enlace = elemento.get("data-item-link")

        imagen_tag = elemento.select_one("img")
        imagen = imagen_tag.get("src") if imagen_tag else None

        # Construir la URL completa de la pelicula
        if enlace and enlace.startswith("/"):
            enlace = "https://letterboxd.com" + enlace

        peliculas.append({
            "posicion": (numero_pagina - 1) * 100 + indice_local,
            "titulo": titulo,
            "link": enlace,
            "imagen": imagen,
        })

    time.sleep(1)

print("Total de peliculas extraidas:", len(peliculas))

Codigo HTTP pagina 1: 200
Paginas encontradas: 3
 - https://letterboxd.com/dave/list/imdb-top-250/
 - https://letterboxd.com/dave/list/imdb-top-250/page/2/
 - https://letterboxd.com/dave/list/imdb-top-250/page/3/
Pagina 1 : 100 peliculas
Codigo HTTP pagina 2 : 200
Pagina 2 : 100 peliculas
Codigo HTTP pagina 3 : 200
Pagina 3 : 50 peliculas
Total de peliculas extraidas: 250


In [2]:
for pelicula in peliculas[:5] + peliculas[-5:]:
    print("Posicion:", pelicula["posicion"])
    print("Titulo:", pelicula["titulo"])
    print("Enlace:", pelicula["link"])
    print("Imagen:", pelicula["imagen"])
    print("-" * 60)

Posicion: 0
Titulo: The Shawshank Redemption (1994)
Enlace: https://letterboxd.com/film/the-shawshank-redemption/
Imagen: https://s.ltrbxd.com/static/img/empty-poster-125-AiuBHVCI.png
------------------------------------------------------------
Posicion: 1
Titulo: The Godfather (1972)
Enlace: https://letterboxd.com/film/the-godfather/
Imagen: https://s.ltrbxd.com/static/img/empty-poster-125-AiuBHVCI.png
------------------------------------------------------------
Posicion: 2
Titulo: The Dark Knight (2008)
Enlace: https://letterboxd.com/film/the-dark-knight/
Imagen: https://s.ltrbxd.com/static/img/empty-poster-125-AiuBHVCI.png
------------------------------------------------------------
Posicion: 3
Titulo: The Godfather Part II (1974)
Enlace: https://letterboxd.com/film/the-godfather-part-ii/
Imagen: https://s.ltrbxd.com/static/img/empty-poster-125-AiuBHVCI.png
------------------------------------------------------------
Posicion: 4
Titulo: The Lord of the Rings: The Return of the King 

In [3]:
import csv

# Exportar todas las peliculas a CSV
with open("peliculas.csv", "w", newline="", encoding="utf-8") as archivo:
    escritor = csv.DictWriter(
        archivo,
        fieldnames=["posicion", "titulo", "link", "imagen"],
    )
    escritor.writeheader()
    escritor.writerows(peliculas)

print(len(peliculas), "peliculas guardadas en peliculas.csv")

250 peliculas guardadas en peliculas.csv


In [4]:

import json
import unicodedata
import re
import pandas as pd

def limpiar(texto):
    if not texto:
        return ""
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

registros = []

for i, pelicula in enumerate(peliculas, start=1):
    url = pelicula["link"]
    response = requests.get(url, headers=headers, timeout=30)
    if response.status_code != 200:
        print(f"[{i}/250] ERROR {response.status_code}: {url}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    bloque = soup.find("script", type="application/ld+json")
    if not bloque or not bloque.string:
        print(f"[{i}/250] Sin JSON-LD: {url}")
        continue

    raw = bloque.string
    datos = json.loads(raw[raw.find("{"):raw.rfind("}") + 1])

    titulo = limpiar(datos.get("name"))
    directores = {limpiar(d["name"]) for d in datos.get("director", []) if d.get("name")}
    generos = {limpiar(g) for g in datos.get("genre", []) if g}
    elenco = {limpiar(a["name"]) for a in datos.get("actor", [])[:12] if a.get("name")}

    registros.append({
        "posicion": pelicula["posicion"],
        "titulo": titulo,
        "directores": directores,
        "generos": generos,
        "elenco": elenco,
    })

    if i % 25 == 0:
        print(f"[{i}/250] procesadas...")

    time.sleep(0.5)

print("Peliculas procesadas:", len(registros))


[25/250] procesadas...
[50/250] procesadas...
[75/250] procesadas...
[100/250] procesadas...
[125/250] procesadas...
[150/250] procesadas...
[175/250] procesadas...
[200/250] procesadas...
[225/250] procesadas...
[250/250] procesadas...
Peliculas procesadas: 250


In [6]:

df = pd.DataFrame(registros)

# Los sets no se pueden escribir directamente al CSV: se serializan como texto
df_csv = df.copy()
for columna in ["directores", "generos", "elenco"]:
    df_csv[columna] = df_csv[columna].apply(lambda s: ", ".join(sorted(s)))

df_csv.to_csv("peliculas.csv", index=False, encoding="utf-8")

print(df_csv.head(3).to_string())
print("...")
print("Filas exportadas:", len(df_csv))


   posicion                    titulo            directores                  generos                                                                                                                                                                                     elenco
0         0  the shawshank redemption        frank darabont             crime, drama            bob gunton, brian libby, clancy brown, gil bellows, james whitmore, jeffrey demunn, larry brandenburg, mark rolston, morgan freeman, neil giuntoli, tim robbins, william sadler
1         1             the godfather  francis ford coppola             crime, drama               al lettieri, al pacino, diane keaton, gianni russo, james caan, john marley, marlon brando, richard conte, richard s castellano, robert duvall, sterling hayden, talia shire
2         2           the dark knight     christopher nolan  action, crime, thriller  aaron eckhart, chin han, christian bale, cillian murphy, gary oldman, heath ledger, maggie gyllenh

In [7]:

import pandas as pd

# Fase 2 (carga): reconstruir los sets desde el CSV
df_peliculas = pd.read_csv("peliculas.csv", encoding="utf-8")

def a_conjunto(texto):
    if pd.isna(texto) or not str(texto).strip():
        return set()
    return set(item.strip() for item in str(texto).split(",") if item.strip())

peliculas_dataset = []
for fila in df_peliculas.to_dict("records"):
    peliculas_dataset.append({
        "posicion": fila["posicion"],
        "titulo": fila["titulo"],
        "directores": a_conjunto(fila["directores"]),
        "generos": a_conjunto(fila["generos"]),
        "elenco": a_conjunto(fila["elenco"]),
    })

print("Peliculas cargadas:", len(peliculas_dataset))
print("Ejemplo:", peliculas_dataset[0]["titulo"], "->", peliculas_dataset[0]["generos"])

# Fase 3: Jaccard ponderado con los pesos asignados
W_GENEROS = 0.5
W_DIRECTORES = 0.3
W_ELENCO = 0.2

def jaccard_ponderado(peli_a, peli_b):

    numerador = 0.0
    denominador = 0.0

    # Generos
    for genero in peli_a["generos"].union(peli_b["generos"]):

        if genero in peli_a["generos"] and genero in peli_b["generos"]:
            numerador += W_GENEROS

        denominador += W_GENEROS

    # Directores
    for director in peli_a["directores"].union(peli_b["directores"]):

        if director in peli_a["directores"] and director in peli_b["directores"]:
            numerador += W_DIRECTORES

        denominador += W_DIRECTORES

    # Elenco
    for actor in peli_a["elenco"].union(peli_b["elenco"]):

        if actor in peli_a["elenco"] and actor in peli_b["elenco"]:
            numerador += W_ELENCO

        denominador += W_ELENCO

    if denominador == 0:
        return 0

    return numerador / denominador


Peliculas cargadas: 250
Ejemplo: the shawshank redemption -> {'crime', 'drama'}


In [8]:

# Fase 4: sistema de recomendacion
def recomendar(titulo_buscado):

    titulo_normalizado = limpiar(titulo_buscado)

    pelicula_buscada = None
    for pelicula in peliculas_dataset:
        if pelicula["titulo"] == titulo_normalizado:
            pelicula_buscada = pelicula
            break

    if pelicula_buscada is None:
        print("La pelicula no existe en el catalogo.")
        return

    print("Pelicula seleccionada:", pelicula_buscada["titulo"])

    similitudes = []

    for pelicula in peliculas_dataset:

        if pelicula["posicion"] != pelicula_buscada["posicion"]:

            similitud = jaccard_ponderado(pelicula_buscada, pelicula)
            similitudes.append((pelicula["titulo"], similitud))

    similitudes.sort(key=lambda par: par[1], reverse=True)

    print()
    print("Top 5 peliculas similares:")
    for i in range(5):
        nombre, valor = similitudes[i]
        print(str(i + 1) + ".", nombre, "->", str(round(valor * 100, 2)) + "%")


In [13]:

recomendar("avengers infinity war")
print()
recomendar("goodfellas")


Pelicula seleccionada: avengers infinity war

Top 5 peliculas similares:
1. avengers endgame -> 76.47%
2. spider man no way home -> 27.94%
3. the empire strikes back -> 20.83%
4. star wars -> 20.83%
5. return of the jedi -> 20.83%

Pelicula seleccionada: goodfellas

Top 5 peliculas similares:
1. casino -> 34.55%
2. taxi driver -> 25.42%
3. once upon a time in america -> 23.33%
4. raging bull -> 23.33%
5. the departed -> 19.7%
